# Project: Yellow Taxi Trip - ETL Pipeline

### Dataset: Yellow Taxi Trip Records - Março 2016

In [1]:
#Realizando as importações das bibliotecas

from pyspark.sql import SparkSession
import pandas as pd
import numpy as np
from pathlib import Path
import kagglehub
import os

In [2]:
# Iniciando instânica Pyspark (pensando em memória local)

spark = (
    SparkSession.builder
    .appName("nyc_taxi_etl")
    .config("spark.driver.memory", "3g")  # ajusta conforme sua RAM disponível
    .config("spark.sql.shuffle.partitions", "8")  # reduz overhead em máquina local
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/08 17:33:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


#### 1. Extract - Leitura dos dados



In [4]:
# Irá pegar o arquivo no Kaggle, ler e salvar no path descrito


path = kagglehub.dataset_download(
    "elemento/nyc-yellow-taxi-trip-data",
    path="yellow_tripdata_2016-03.csv")

print(f'Arquivo salvo em: \n\n {path}')

# OBS: Caso queira apagar depois, só ir pra esse caminho e apagar manualmente

# Iniando o dataframe:

df = spark.read.csv(
    path,  # o path que o kagglehub te deu
    header=True,
    inferSchema=True  # cuidado: em CSVs grandes isso é mais lento, ver alternativa abaixo
)

print("DataFrame criado!")

Arquivo salvo em: 

 /home/nicolinux/.cache/kagglehub/datasets/elemento/nyc-yellow-taxi-trip-data/versions/2/yellow_tripdata_2016-03.csv


DataFrame criado!


In [6]:
# Analisando estrutura do Df

print(f"Número de linhas: {df.count()} \nNúmero de colunas: {len(df.columns)}\n")


print("Vendo o HEAD do df:")

df.show(5, truncate=False)


Número de linhas: 12210952 
Número de colunas: 19

Vendo o HEAD do df:
+--------+--------------------+---------------------+---------------+-------------+------------------+------------------+----------+------------------+------------------+-----------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|pickup_longitude  |pickup_latitude   |RatecodeID|store_and_fwd_flag|dropoff_longitude |dropoff_latitude |payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|
+--------+--------------------+---------------------+---------------+-------------+------------------+------------------+----------+------------------+------------------+-----------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|1       |2016-03-01 00:00:00 |2016-03-01 00:07:55  |1              

In [ ]:
# Analisando a estrutura do df

print("Tipos dos dados:")

df.printSchema()


# Coerção de datas

from pyspark.sql.functions import to_timestamp

df = (
    df
    .withColumn("pickup_datetime", to_timestamp("tpep_pickup_datetime"))
    .withColumn("dropoff_datetime", to_timestamp("tpep_dropoff_datetime")) 
)


Tipos dos dados:
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)



#### 2. Tratamentos dos dados

In [9]:
# Identificação de valores nulos:

from pyspark.sql.functions import col, sum as spark_sum, when

null_counts = df.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
])

null_counts.show()

+--------+--------------------+---------------------+---------------+-------------+----------------+---------------+----------+------------------+-----------------+----------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+---------------+----------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|pickup_longitude|pickup_latitude|RatecodeID|store_and_fwd_flag|dropoff_longitude|dropoff_latitude|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|pickup_datetime|dropoff_datetime|
+--------+--------------------+---------------------+---------------+-------------+----------------+---------------+----------+------------------+-----------------+----------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+---------------+----------------+
|       0|                   0|                    0|        

In [ ]:
# Identificação de anomalias


# 1 - número de passageiros invalido

passageiros_invalidos = (df['passenger_count'] <= 0) | (df['passenger_count'] > 6)


# distancias maiores que o normal (ou menores que o normal)

distancias_invalidas = (df['trip_distance'] <= 0) | (df['trip_distance'] > 100)


# hora de saida menor que a hora de entrada

horas_invalidas = df['tpep_dropoff_datetime'] <= df['tpep_pickup_datetime']



#tarifa negativa

tarifa_negativa = df['fare_amount'] < 0


# corridas fora de NYC

coords_entrada_invalidas = (
    (df['pickup_latitude'] < 40.4) | (df['pickup_latitude'] > 41.0) |
    (df['pickup_longitude'] < -74.3) | (df['pickup_longitude'] > -73.7)
) 

coords_saida_invalidas =  (
    (df['dropoff_latitude'] < 40.4) | (df['dropoff_latitude'] > 41.0) |
    (df['dropoff_longitude'] < -74.3) | (df['dropoff_longitude'] > -73.3)
)


coord = (
    (df['pickup_latitude'] < 40.4) | (df['pickup_latitude'] > 41.0) |
    (df['pickup_longitude'] < -74.3) | (df['pickup_longitude'] > -73.7)
) | (
    (df['dropoff_latitude'] < 40.4) | (df['dropoff_latitude'] > 41.0) |
    (df['dropoff_longitude'] < -74.3) | (df['dropoff_longitude'] > -73.3)
)






# RESULTADO DAS ANOMALIAS:

print(f"Quantidade de passageiros invalidos: {passageiros_invalidos.sum()} ({passageiros_invalidos.sum()/df.shape[0] * 100: .5f} %)")
print(f"Quantidade de distancias invalidas: {distancias_invalidas.sum()} ({distancias_invalidas.sum()/df.shape[0] * 100: .5f} %)")
print(f"Quantidade de horas invalidas: {horas_invalidas.sum()} ({horas_invalidas.sum()/df.shape[0] * 100: .5f} %)")
print(f"Quantidade de tarifas invalidas: {tarifa_negativa.sum()} ({tarifa_negativa.sum()/df.shape[0] *100: .5f} %)")
print(f"Quantidade de pickup fora de NYC: {coords_entrada_invalidas.sum()} ({coords_entrada_invalidas.sum()/df.shape[0] *100: .5f} %)")
print(f"Quantidade de dropoff fora de NYC: {coords_saida_invalidas.sum()} ({coords_saida_invalidas.sum()/df.shape[0] *100: .5f} %)")



Quantidade de passageiros invalidos: 678 ( 0.00555 %)
Quantidade de distancias invalidas: 71225 ( 0.58329 %)
Quantidade de horas invalidas: 12550 ( 0.10278 %)
Quantidade de tarifas invalidas: 4581 ( 0.03752 %)
Quantidade de pickup fora de NYC: 183816 ( 1.50534 %)
Quantidade de dropoff fora de NYC: 174501 ( 1.42905 %)


In [ ]:
#Retirando os registros invalidos do DF

filtro_invalidos = (
    passageiros_invalidos |
    distancias_invalidas |
    horas_invalidas |
    tarifa_negativa |
    coords_entrada_invalidas |
    coords_saida_invalidas 
)

df_filtrado = df[~filtro_invalidos].copy()

linhas_removidas = df.shape[0] - df_filtrado.shape[0]

print(f"Quantidade de linhas antes: {df.shape[0]}")


print(f"Quantidade de linhas removidas: {linhas_removidas} ({linhas_removidas/df.shape[0] *100: .2f} %)")

print(f"Quantidade de linhas depois: {df_filtrado.shape[0]}")

#### 3. Enriquecimento dos dados

#### 4. Agregações

#### 5. Load